# 🧠 Baseline Feature Extraction (Query & Gallery)

Notebook ini mengekstrak fitur visual menggunakan model *pre-trained* (**ImageNet, tanpa fine-tuning**) khusus untuk subset **Query** dan **Gallery** dari `full_dataset.csv` guna keperluan evaluasi *retrieval*.

In [3]:
import os, sys, gc, time
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.applications import ResNet50, VGG19, InceptionV3, MobileNetV3Large
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D

print(f"✅ TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ Memory growth enabled for {len(gpus)} GPU(s)")
else:
    print("⚠️ GPU tidak terdeteksi, berjalan di CPU.")

✅ TensorFlow version: 2.10.0
✅ Memory growth enabled for 1 GPU(s)


In [9]:
# ── Path Setup (Berdasarkan Sistem Operasi Kernel Python) ───
if sys.platform == 'win32':
    BASE_DIR = r"d:\Antigravity\Visual Based Rekomender Sistem"
    print(f"🪟 Kernel berjalan di: Windows Native ({BASE_DIR})")
else:
    BASE_DIR = '/mnt/d/Antigravity/Visual Based Rekomender Sistem'
    print(f"🐧 Kernel berjalan di: Linux / WSL2 ({BASE_DIR})")

FEAT_DIR = os.path.join(BASE_DIR, 'features', 'exp1_baseline')
FULL_CSV = os.path.join(BASE_DIR, 'skripsi', 'persiapan_skripsi', 'data preperation', 'dataset_output', 'full_dataset.csv')
IMG_DIR  = os.path.join(BASE_DIR, 'dataset', 'In-shop Clothes Retrieval Benchmark', 'Img')

os.makedirs(FEAT_DIR, exist_ok=True)

# ── Konfigurasi Model Baseline ───────────────────────────────
MODEL_CONFIGS = {
    'resnet50': {
        'class': ResNet50,
        'input_size': (224, 224),
        'preprocess': tf.keras.applications.resnet50.preprocess_input,
    },
    'vgg19': {
        'class': VGG19,
        'input_size': (224, 224),
        'preprocess': tf.keras.applications.vgg19.preprocess_input,
    },
    'inceptionv3': {
        'class': InceptionV3,
        'input_size': (299, 299),
        'preprocess': tf.keras.applications.inception_v3.preprocess_input,
    },
    'mobilenetv3': {
        'class': MobileNetV3Large,
        'input_size': (224, 224),
        'preprocess': tf.keras.applications.mobilenet_v3.preprocess_input,
    },
}

BATCH_SIZE = 128
print("✅ Path & konfigurasi model berhasil di-load.")

🪟 Kernel berjalan di: Windows Native (d:\Antigravity\Visual Based Rekomender Sistem)
✅ Path & konfigurasi model berhasil di-load.


In [10]:
# Load full_dataset.csv
df_full = pd.read_csv(FULL_CSV)

# Verifikasi full_path gambar secara dinamis
df_full['full_path'] = df_full['image_name'].apply(
    lambda x: os.path.join(IMG_DIR, os.path.normpath(str(x).replace('img/', '', 1)))
)

# Filter hanya subset query dan gallery
df_query   = df_full[df_full['split'] == 'query'].reset_index(drop=True)
df_gallery = df_full[df_full['split'] == 'gallery'].reset_index(drop=True)

print("=" * 55)
print("  📊 DATASET RETRIEVAL BASELINE (EXP 1)")
print("=" * 55)
print(f"  Total QUERY   : {len(df_query):,}")
print(f"  Total GALLERY : {len(df_gallery):,}")
print("=" * 55)

  📊 DATASET RETRIEVAL BASELINE (EXP 1)
  Total QUERY   : 14,218
  Total GALLERY : 12,612


In [ ]:
# ── Dataset Statistics & Overview ────────────────────────────
print("\n" + "=" * 55)
print("  📊 DATASET STATISTICS (QUERY & GALLERY)")
print("=" * 55)

# Kategori unik di query dan gallery
query_cats   = df_query['category'].unique()
gallery_cats = df_gallery['category'].unique()

print(f"\n  📌 QUERY SET:")
print(f"     Total gambar    : {len(df_query):,}")
print(f"     Kategori unik   : {len(query_cats)}")
print(f"     Kategori        : {', '.join(sorted(query_cats))}")

print(f"\n  📌 GALLERY SET:")
print(f"     Total gambar    : {len(df_gallery):,}")
print(f"     Kategori unik   : {len(gallery_cats)}")
print(f"     Kategori        : {', '.join(sorted(gallery_cats))}")

# Distribusi per kategori
print(f"\n  📈 DISTRIBUSI PER KATEGORI:")
print(f"  {'Kategori':<20} {'Query':>8} {'Gallery':>10}")
print(f"  {'-'*42}")

all_cats = sorted(set(query_cats) | set(gallery_cats))
for cat in all_cats:
    q_count = len(df_query[df_query['category'] == cat])
    g_count = len(df_gallery[df_gallery['category'] == cat])
    print(f"  {cat:<20} {q_count:>8,} {g_count:>10,}")

# Verifikasi path gambar
missing_query   = df_query[~df_query['full_path'].apply(os.path.exists)]
missing_gallery = df_gallery[~df_gallery['full_path'].apply(os.path.exists)]

print(f"\n  ⚠️ GAMBAR HILANG:")
print(f"     Query   : {len(missing_query):,} dari {len(df_query):,} ({len(missing_query)/len(df_query)*100:.1f}%)")
print(f"     Gallery : {len(missing_gallery):,} dari {len(df_gallery):,} ({len(missing_gallery)/len(df_gallery)*100:.1f}%)")

if len(missing_query) > 0:
    print(f"\n  📋 Contoh path yang hilang (Query):")
    for _, row in missing_query.head(3).iterrows():
        print(f"     - {row['full_path']}")

print("\n✅ Statistik dataset selesai!")

In [11]:
def l2_normalize(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1e-10, norms)
    return matrix / norms

def extract_features(model, image_paths, input_size, preprocess_fn, desc='Extracting'):
    features = []
    valid_idx = []

    for start in tqdm(range(0, len(image_paths), BATCH_SIZE), desc=desc):
        batch_paths = image_paths[start:start + BATCH_SIZE]
        batch_imgs = []
        batch_indices = []

        for i, path in enumerate(batch_paths):
            try:
                img = Image.open(path).convert('RGB').resize(input_size)
                arr = np.array(img, dtype=np.float32)
                arr = np.expand_dims(arr, axis=0)
                arr = preprocess_fn(arr)
                batch_imgs.append(arr)
                batch_indices.append(start + i)
            except Exception:
                continue

        if not batch_imgs:
            continue

        batch_array = np.vstack(batch_imgs)
        batch_feats = model.predict(batch_array, verbose=0)
        features.append(batch_feats)
        valid_idx.extend(batch_indices)

    if not features:
        return np.array([]), []

    feat_matrix = np.vstack(features)
    feat_matrix = l2_normalize(feat_matrix)
    return feat_matrix, valid_idx

print("✅ Helper functions siap.")

✅ Helper functions siap.


In [12]:
query_paths   = df_query['full_path'].tolist()
gallery_paths = df_gallery['full_path'].tolist()

for model_name, cfg in MODEL_CONFIGS.items():
    print(f"\n{'='*55}")
    print(f"  🧠 BASELINE (ImageNet) — {model_name.upper()}")
    print(f"{'='*55}")

    q_feat_path = os.path.join(FEAT_DIR, f'{model_name}_query_features.npy')
    g_feat_path = os.path.join(FEAT_DIR, f'{model_name}_gallery_features.npy')
    q_idx_path  = os.path.join(FEAT_DIR, f'{model_name}_query_valid_idx.npy')
    g_idx_path  = os.path.join(FEAT_DIR, f'{model_name}_gallery_valid_idx.npy')

    if os.path.exists(q_feat_path) and os.path.exists(g_feat_path):
        print(f"  ⏩ Fitur {model_name} sudah ada, skip!")
        continue

    base_model = cfg['class'](weights='imagenet', include_top=False, input_shape=cfg['input_size'] + (3,))
    x = GlobalAveragePooling2D()(base_model.output)
    model = Model(inputs=base_model.input, outputs=x)

    print(f"  📌 Extracting QUERY ({len(query_paths):,} images)...")
    q_feats, q_valid = extract_features(
        model, query_paths, cfg['input_size'], cfg['preprocess'],
        desc=f"Query [{model_name}]"
    )

    print(f"  📌 Extracting GALLERY ({len(gallery_paths):,} images)...")
    g_feats, g_valid = extract_features(
        model, gallery_paths, cfg['input_size'], cfg['preprocess'],
        desc=f"Gallery [{model_name}]"
    )

    np.save(q_feat_path, q_feats)
    np.save(g_feat_path, g_feats)
    np.save(q_idx_path, np.array(q_valid))
    np.save(g_idx_path, np.array(g_valid))

    print(f"  ✅ Query features shape  : {q_feats.shape}")
    print(f"  ✅ Gallery features shape: {g_feats.shape}")

    del model, base_model, q_feats, g_feats
    gc.collect()
    tf.keras.backend.clear_session()

print("\n🎉 EKSTRAKSI FITUR BASELINE RETRIEVAL SELESAI!")


  🧠 BASELINE (ImageNet) — RESNET50
  📌 Extracting QUERY (14,218 images)...


Query [resnet50]: 100%|██████████| 112/112 [01:40<00:00,  1.11it/s]


  📌 Extracting GALLERY (12,612 images)...


Gallery [resnet50]: 100%|██████████| 99/99 [01:57<00:00,  1.19s/it]


  ✅ Query features shape  : (14218, 2048)
  ✅ Gallery features shape: (12612, 2048)

  🧠 BASELINE (ImageNet) — VGG19
  📌 Extracting QUERY (14,218 images)...


...

  ✅ Query features shape  : (14218, 2048)
  ✅ Gallery features shape: (12612, 2048)

  🧠 BASELINE (ImageNet) — MOBILENETV3
  📌 Extracting QUERY (14,218 images)...


Query [mobilenetv3]: 100%|██████████| 112/112 [00:34<00:00,  3.21it/s]


  📌 Extracting GALLERY (12,612 images)...


Gallery [mobilenetv3]: 100%|██████████| 99/99 [00:29<00:00,  3.37it/s]


  ✅ Query features shape  : (14218, 960)
  ✅ Gallery features shape: (12612, 960)

🎉 EKSTRAKSI FITUR BASELINE RETRIEVAL SELESAI!
